# Bitcoin On-Chain Data — Blockchain.com Charts API

Este notebook constrói um dataset histórico completo de métricas on-chain do Bitcoin usando a **API Charts do Blockchain.com** (gratuita, sem autenticação), enriquecido com preço (CoinGecko) e métricas derivadas.

## Pipeline

1. **Coleta base** — 6 métricas on-chain via API (transações, endereços, hashrate, dificuldade, receita, volume BTC)
2. **Enriquecimento** — preço BTC/USD (CoinGecko + API fallback), volume USD, UTXO count, total BTC
3. **Métricas derivadas** — criação de UTXOs, custo de produção (proxy receita + modelo eletricidade via hashrate)
4. **Exportação** — CSV base + CSV enriquecido

## Métricas Coletadas (Base)

| Métrica API | Coluna | Descrição |
|---|---|---|
| `n-transactions` | `transactions` | Transações confirmadas / dia |
| `n-unique-addresses` | `active_addresses` | Endereços únicos ativos / dia |
| `hash-rate` | `hashrate` | Poder computacional (TH/s) |
| `difficulty` | `difficulty` | Dificuldade de mineração |
| `miners-revenue` | `miners_revenue` | Receita dos mineradores (USD/dia) |
| `estimated-transaction-volume` | `transaction_volume_btc` | Volume estimado (BTC/dia) |

## Métricas Enriquecidas

| Fonte | Coluna | Descrição |
|---|---|---|
| CoinGecko + API | `price_usd` | Preço BTC/USD diário |
| API | `transaction_volume_usd` | Volume on-chain em USD |
| API | `utxo_count` | Total de UTXOs |
| API | `total_btc` | Total de BTC em circulação |
| API | `fees_usd` | Taxas de transação pagas aos mineradores (USD/dia) |
| Derivada | `block_rewards_usd` | Recompensa de bloco em USD (receita − taxas) |
| Derivada | `utxo_creation` | Δ diário UTXOs |
| Derivada | `cost_per_btc` | Receita / BTC minerado (proxy) |
| Derivada | `electricity_cost_per_btc` | Custo eletricidade / BTC (modelo hashrate) |

## Fonte de Dados


- **API**: https://api.blockchain.info/charts/{chartName}- **CoinGecko**: `files/coingecko/btc-usd-max.csv`

In [12]:
import requests
import pandas as pd
import time
from datetime import datetime

## 1. Configuração

Definição das métricas a coletar, período e parâmetros da API.

In [13]:
BASE_URL = "https://api.blockchain.info/charts"
START_DATE = "2015-01-01"
OUTPUT_DIR = "files/blockchain_com"
OUTPUT_FILE = f"{OUTPUT_DIR}/btc_onchain_daily.csv"

# Mapeamento: nome_api -> nome_coluna_limpo
METRICS = {
    "n-transactions":                "transactions",
    "n-unique-addresses":            "active_addresses",
    "hash-rate":                     "hashrate",
    "difficulty":                    "difficulty",
    "miners-revenue":                "miners_revenue",
    "estimated-transaction-volume":  "transaction_volume_btc",
}

print(f"Métricas: {len(METRICS)}")
print(f"Período: {START_DATE} até hoje")
print(f"Output: {OUTPUT_FILE}")

Métricas: 6
Período: 2015-01-01 até hoje
Output: files/blockchain_com/btc_onchain_daily.csv


## 2. Coleta de Dados

Função que consulta a API Charts do Blockchain.com para cada métrica.

**Parâmetros importantes da API:**
- `timespan`: duração do gráfico (ex: `"12years"`)
- `start`: data inicial no formato `YYYY-MM-DD`
- `sampled=false`: retorna **todos** os pontos de dados (sem amostragem)
- `format=json`: resposta em JSON com array `values[{x, y}]`

In [14]:
def fetch_blockchain_chart(chart_name: str, start: str = START_DATE, timespan: str = "12years") -> pd.DataFrame:
    """
    Baixa dados de uma métrica da API Charts do Blockchain.com.
    Retorna DataFrame com colunas ['date', chart_name].
    """
    url = f"{BASE_URL}/{chart_name}"
    params = {
        "timespan": timespan,
        "start": start,
        "sampled": "false",
        "format": "json",
    }

    print(f"  Fetching {chart_name}...", end=" ")
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()

    values = data.get("values", [])
    df = pd.DataFrame(values)
    df.columns = ["timestamp", "value"]

    # Converter Unix timestamp para date
    df["date"] = pd.to_datetime(df["timestamp"], unit="s").dt.date
    df["date"] = pd.to_datetime(df["date"])

    # Remover duplicatas (manter último valor do dia)
    df = df.drop_duplicates(subset="date", keep="last")
    df = df[["date", "value"]].sort_values("date").reset_index(drop=True)

    print(f"OK — {len(df)} registros ({df['date'].min().date()} a {df['date'].max().date()})")
    return df

In [15]:
raw_data = {}

print("Baixando métricas da Blockchain.com...\n")
for api_name, col_name in METRICS.items():
    try:
        df = fetch_blockchain_chart(api_name)
        df = df.rename(columns={"value": col_name})
        raw_data[col_name] = df
        time.sleep(1)  # rate limiting — respeitar o servidor
    except requests.exceptions.RequestException as e:
        print(f"  ERRO ao baixar {api_name}: {e}")

print(f"\nMétricas baixadas com sucesso: {len(raw_data)}/{len(METRICS)}")

Baixando métricas da Blockchain.com...

  Fetching n-transactions... OK — 4110 registros (2015-01-01 a 2026-04-05)
  Fetching n-unique-addresses... OK — 4107 registros (2015-01-01 a 2026-04-05)
  Fetching hash-rate... OK — 4110 registros (2015-01-01 a 2026-04-05)
  Fetching difficulty... OK — 4110 registros (2015-01-01 a 2026-04-05)
  Fetching miners-revenue... OK — 4113 registros (2015-01-01 a 2026-04-05)
  Fetching estimated-transaction-volume... OK — 4105 registros (2015-01-01 a 2026-04-05)

Métricas baixadas com sucesso: 6/6


## 3. Unificação dos Dados

Merge de todas as métricas em um único DataFrame usando a coluna `date` como chave. Garante frequência diária completa via reindex.

In [16]:
# Merge sequencial de todos os DataFrames pela coluna 'date'
dfs = list(raw_data.values())
df_merged = dfs[0]

for df in dfs[1:]:
    df_merged = pd.merge(df_merged, df, on="date", how="outer")

df_merged = df_merged.sort_values("date").reset_index(drop=True)

# Garantir frequência diária completa (reindex com range de datas)
date_range = pd.date_range(start=df_merged["date"].min(), end=df_merged["date"].max(), freq="D")
df_merged = df_merged.set_index("date").reindex(date_range)
df_merged.index.name = "date"
df_merged = df_merged.reset_index()

print(f"Shape após merge: {df_merged.shape}")
print(f"Período: {df_merged['date'].min().date()} a {df_merged['date'].max().date()}")
print(f"\nValores faltantes por coluna:")
print(df_merged.isnull().sum())

Shape após merge: (4113, 7)
Período: 2015-01-01 a 2026-04-05

Valores faltantes por coluna:
date                      0
transactions              3
active_addresses          6
hashrate                  3
difficulty                3
miners_revenue            0
transaction_volume_btc    8
dtype: int64


## 4. Tratamento de Valores Faltantes

Estratégia:
1. **Forward fill** (`ffill`): propaga o último valor válido para frente — adequado para métricas como `difficulty` e `hashrate` que mudam discretamente.
2. **Backward fill** (`bfill`): preenche eventuais NaN no início do dataset (antes do primeiro valor disponível).
3. **Verificação final** de que não restam valores nulos.

In [17]:
numeric_cols = [c for c in df_merged.columns if c != "date"]

df_merged[numeric_cols] = df_merged[numeric_cols].ffill()
df_merged[numeric_cols] = df_merged[numeric_cols].bfill()

remaining_nulls = df_merged.isnull().sum().sum()
print(f"Valores faltantes restantes: {remaining_nulls}")
print(f"\nShape final: {df_merged.shape}")
df_merged.dtypes

Valores faltantes restantes: 0

Shape final: (4113, 7)


date                      datetime64[ns]
transactions                     float64
active_addresses                 float64
hashrate                         float64
difficulty                       float64
miners_revenue                   float64
transaction_volume_btc           float64
dtype: object

## 5. Exportação do Dataset Base

In [18]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)
df_merged.to_csv(OUTPUT_FILE, index=False)

file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f"Arquivo salvo: {OUTPUT_FILE}")
print(f"Tamanho: {file_size_mb:.2f} MB")
print(f"Linhas: {len(df_merged):,}")
print(f"Colunas: {list(df_merged.columns)}")

Arquivo salvo: files/blockchain_com/btc_onchain_daily.csv
Tamanho: 0.39 MB
Linhas: 4,113
Colunas: ['date', 'transactions', 'active_addresses', 'hashrate', 'difficulty', 'miners_revenue', 'transaction_volume_btc']


## 6. Preview do Dataset

In [19]:
print("=== Primeiras linhas ===")
display(df_merged.head(10))

print("\n=== Últimas linhas ===")
display(df_merged.tail(10))

=== Primeiras linhas ===


,date,transactions,active_addresses,hashrate,difficulty,miners_revenue,transaction_volume_btc
0,2015-01-01,59344.0,117529.0,335365.290092,4.064096e+10,1.519572e+06,80738.372894
1,2015-01-02,79287.0,179406.0,323243.653101,4.064096e+10,1.475784e+06,131111.311672
2,2015-01-03,82227.0,205926.0,331324.744428,4.064096e+10,1.586907e+06,285476.171649
3,2015-01-04,85694.0,207255.0,335365.290092,4.064096e+10,1.632203e+06,289732.768007
4,2015-01-05,95585.0,200612.0,339405.835756,4.064096e+10,1.202039e+06,259995.254455
5,2015-01-06,87821.0,179159.0,317182.834605,4.064096e+10,1.361861e+06,294497.543721
6,2015-01-07,89916.0,171674.0,299000.379118,4.064096e+10,1.052470e+06,227982.464014
7,2015-01-08,107132.0,197689.0,272736.832304,4.064096e+10,1.383767e+06,332357.348123
8,2015-01-09,102275.0,202190.0,307081.470446,4.064096e+10,1.085524e+06,241529.314206
9,2015-01-10,104874.0,242729.0,284858.469295,4.064096e+10,1.335502e+06,213685.502774



=== Últimas linhas ===


,date,transactions,active_addresses,hashrate,difficulty,miners_revenue,transaction_volume_btc
4103,2026-03-27,664609.0,507433.0,1.010936e+09,1.337931e+14,3.257280e+07,129836.582986
4104,2026-03-28,763304.0,432914.0,1.157255e+09,1.337931e+14,3.754956e+07,40732.552955
4105,2026-03-29,634952.0,382086.0,9.577287e+08,1.337931e+14,3.133210e+07,29736.944100
4106,2026-03-30,532760.0,459981.0,9.045215e+08,1.337931e+14,3.165220e+07,108746.419864
4107,2026-03-31,523751.0,485131.0,9.045215e+08,1.337931e+14,2.920899e+07,109013.092648
4108,2026-04-01,683419.0,527189.0,1.077445e+09,1.337931e+14,3.694607e+07,94730.429643
4109,2026-04-02,508404.0,505693.0,9.643795e+08,1.337931e+14,3.294413e+07,100465.141306
4110,2026-04-03,459037.0,498568.0,8.987501e+08,1.369679e+14,2.856551e+07,45034.892274
4111,2026-04-04,433372.0,430348.0,8.289697e+08,1.389669e+14,2.698169e+07,21521.247465
4112,2026-04-05,725796.0,380100.0,1.098385e+09,1.389669e+14,3.644647e+07,21730.706644


In [20]:
print("=== Estatísticas Descritivas ===")
display(df_merged.describe())

=== Estatísticas Descritivas ===


,date,transactions,active_addresses,hashrate,difficulty,miners_revenue,transaction_volume_btc
count,4113,4113.000000,4.113000e+03,4.113000e+03,4.113000e+03,4.113000e+03,4113.000000
mean,2020-08-18 00:00:00,310982.642110,5.393622e+05,2.484029e+08,3.435004e+13,2.258330e+07,164675.324335
min,2015-01-01 00:00:00,59344.000000,1.175290e+05,2.295135e+05,4.064096e+10,6.728124e+05,17263.428639
25%,2017-10-25 00:00:00,231054.000000,4.519650e+05,9.605401e+06,1.196793e+12,7.012104e+06,96302.959654
50%,2020-08-18 00:00:00,288495.000000,5.425080e+05,1.176301e+08,1.610481e+13,1.840677e+07,138580.239249
75%,2023-06-12 00:00:00,359522.000000,6.455830e+05,3.643296e+08,5.064621e+13,3.616277e+07,214175.749256
max,2026-04-05 00:00:00,927010.000000,1.072862e+06,1.305500e+09,1.559730e+14,1.077559e+08,878528.521428
std,NaN,128283.999986,1.468094e+05,3.118916e+08,4.306971e+13,1.807343e+07,102462.271909


---

## 7. Enriquecimento — Preço BTC e Métricas Adicionais

O preço BTC/USD é carregado do CoinGecko (`btc-usd-max.csv`, diário, 2013–2025).  
Para o período mais recente, complementamos com a API `market-price` da Blockchain.com.

Métricas adicionais buscadas da API:
- `estimated-transaction-volume-usd` → volume on-chain em USD
- `utxo-count` → total de UTXOs (para calcular criação diária)
- `total-bitcoins` → total BTC em circulação (para calcular BTC minerados/dia)

In [21]:
import numpy as np

# --- Preço BTC: CoinGecko (arquivo local) como base ---
COINGECKO_PATH = "files/coingecko/btc-usd-max.csv"

cg = pd.read_csv(COINGECKO_PATH)
cg["date"] = pd.to_datetime(cg["snapped_at"], utc=True).dt.normalize().dt.tz_localize(None)
cg = cg[["date", "price"]].rename(columns={"price": "price_usd"})
cg = cg.drop_duplicates(subset="date", keep="last").sort_values("date").reset_index(drop=True)

print(f"CoinGecko: {len(cg)} dias ({cg['date'].min().date()} a {cg['date'].max().date()})")

# Merge preço no dataset base
df = df_merged.copy()
df = pd.merge(df, cg, on="date", how="left")

missing_price = df["price_usd"].isnull().sum()
print(f"Datas sem preço após CoinGecko: {missing_price}")

# --- API Blockchain.com: preço (fallback) + 3 métricas adicionais ---
EXTRA_METRICS = {
    "estimated-transaction-volume-usd": "transaction_volume_usd",
    "utxo-count":                       "utxo_count",
    "total-bitcoins":                   "total_btc",
    "transaction-fees-usd":             "fees_usd",
}
if missing_price > 0:
    EXTRA_METRICS["market-price"] = "price_usd_api"

print(f"\nBaixando {len(EXTRA_METRICS)} métricas extras da API...\n")
for api_name, col_name in EXTRA_METRICS.items():
    try:
        extra = fetch_blockchain_chart(api_name)
        extra = extra.rename(columns={"value": col_name})
        df = pd.merge(df, extra, on="date", how="left")
        time.sleep(1)
    except requests.exceptions.RequestException as e:
        print(f"  ERRO: {api_name} — {e}")

# Preencher gaps de preço com dados da API (se buscados)
if "price_usd_api" in df.columns:
    df["price_usd"] = df["price_usd"].fillna(df["price_usd_api"])
    df = df.drop(columns=["price_usd_api"])
    print(f"\nPreço: gaps preenchidos via API — restam {df['price_usd'].isnull().sum()} nulos")

print(f"\nShape após enriquecimento: {df.shape}")
print(f"Colunas: {list(df.columns)}")

CoinGecko: 4724 dias (2013-04-28 a 2026-04-05)
Datas sem preço após CoinGecko: 1

Baixando 5 métricas extras da API...

  Fetching estimated-transaction-volume-usd... OK — 4105 registros (2015-01-01 a 2026-04-05)
  Fetching utxo-count... OK — 4113 registros (2015-01-01 a 2026-04-06)
  Fetching total-bitcoins... OK — 4114 registros (2015-01-01 a 2026-04-06)
  Fetching transaction-fees-usd... OK — 4113 registros (2015-01-01 a 2026-04-05)
  Fetching market-price... OK — 4114 registros (2015-01-01 a 2026-04-06)

Preço: gaps preenchidos via API — restam 0 nulos

Shape após enriquecimento: (4113, 12)
Colunas: ['date', 'transactions', 'active_addresses', 'hashrate', 'difficulty', 'miners_revenue', 'transaction_volume_btc', 'price_usd', 'transaction_volume_usd', 'utxo_count', 'total_btc', 'fees_usd']


## 8. Métricas Derivadas

Métricas calculadas a partir dos dados brutos:

- **`utxo_creation`**: Δ diário do UTXO set
- **`block_rewards_usd`**: receita total − taxas de transação (decomposição)
- **`cost_per_btc`** (proxy receita): `miners_revenue / btc_minerados_dia`
- **`electricity_cost_per_btc`** (modelo hashrate): custo de eletricidade estimado via eficiência dos ASICs

$$\text{Custo}_{elec} = \frac{\text{Hashrate (TH/s)} \times \text{Eficiência (J/TH)} \times 86{,}400\text{s}}{3{,}6 \times 10^6} \times \$0{,}05/\text{kWh}$$

In [23]:
# Criação líquida de UTXOs por dia
df["utxo_creation"] = df["utxo_count"].diff()

# BTC minerados por dia
df["daily_btc_mined"] = df["total_btc"].diff()

# --- Decomposição da receita: block rewards vs fees ---
df["block_rewards_usd"] = df["miners_revenue"] - df["fees_usd"]
df["block_rewards_usd"] = df["block_rewards_usd"].clip(lower=0)

# --- Proxy 1: Custo baseado em receita ---
df["cost_per_btc"] = df["miners_revenue"] / df["daily_btc_mined"].replace(0, np.nan)

# --- Proxy 2: Custo de eletricidade (modelo hashrate + ASIC) ---
ASIC_EFFICIENCY = [
    ("2015-01-01", 500),   # Antminer S5 / era pré-S9
    ("2016-07-01", 200),   # Antminer S7 / transição
    ("2017-06-01", 100),   # Antminer S9 domina
    ("2019-06-01",  65),   # Antminer S17 / Whatsminer M20S
    ("2020-06-01",  42),   # Antminer S19 / M30S
    ("2021-06-01",  34),   # S19 Pro / M30S++
    ("2022-06-01",  30),   # S19 XP / M50
    ("2023-06-01",  25),   # S19 XP Hyd / S21 early
    ("2024-06-01",  18),   # Antminer S21 / T21
    ("2025-06-01",  15),   # S21 Hyd / next-gen
]

eff_dates = pd.to_datetime([e[0] for e in ASIC_EFFICIENCY])
eff_values = [e[1] for e in ASIC_EFFICIENCY]
df["asic_efficiency_jth"] = np.interp(
    df["date"].astype(np.int64), eff_dates.astype(np.int64), eff_values
)

ELECTRICITY_PRICE = 0.05  # USD/kWh (ref. CBECI)

# Energia diária (kWh) = hashrate (TH/s) × eficiência (J/TH) × 86400s / 3.6e6 (J→kWh)
df["daily_electricity_kwh"] = df["hashrate"] * df["asic_efficiency_jth"] * 86_400 / 3.6e6
df["daily_electricity_cost"] = df["daily_electricity_kwh"] * ELECTRICITY_PRICE
df["electricity_cost_per_btc"] = df["daily_electricity_cost"] / df["daily_btc_mined"].replace(0, np.nan)

# Forward fill NaN pontuais
for col in ["utxo_creation", "cost_per_btc", "electricity_cost_per_btc"]:
    df[col] = df[col].ffill().bfill()

print("Métricas derivadas:")
print(f"  utxo_creation          — min: {df['utxo_creation'].min():.0f}, max: {df['utxo_creation'].max():.0f}")

print(f"  cost_per_btc (receita) — min: ${df['cost_per_btc'].min():,.0f}, max: ${df['cost_per_btc'].max():,.0f}")
print(f"  preço eletricidade     — ${ELECTRICITY_PRICE}/kWh")

print(f"  electricity_cost/btc   — min: ${df['electricity_cost_per_btc'].min():,.0f}, max: ${df['electricity_cost_per_btc'].max():,.0f}")
print(f"  asic_efficiency (J/TH) — {df['asic_efficiency_jth'].iloc[-1]:.0f} (atual)")

Métricas derivadas:
  utxo_creation          — min: -954068, max: 1716197
  cost_per_btc (receita) — min: $193, max: $263,221
  preço eletricidade     — $0.05/kWh
  electricity_cost/btc   — min: $42, max: $47,046
  asic_efficiency (J/TH) — 15 (atual)


## 9. Exportação do Dataset Enriquecido

Salvamos o DataFrame final com todas as colunas — base + preço + métricas adicionais da API + métricas derivadas — em CSV para consumo pelo notebook de gráficos.

In [24]:
ENRICHED_FILE = f"{OUTPUT_DIR}/btc_onchain_enriched.csv"

df.to_csv(ENRICHED_FILE, index=False)
print(f"Dataset enriquecido exportado: {ENRICHED_FILE}")
print(f"  Linhas: {len(df):,}")
print(f"  Colunas ({df.shape[1]}): {', '.join(df.columns)}")
print(f"  Período: {df['date'].min()} a {df['date'].max()}")

Dataset enriquecido exportado: files/blockchain_com/btc_onchain_enriched.csv
  Linhas: 4,113
  Colunas (20): date, transactions, active_addresses, hashrate, difficulty, miners_revenue, transaction_volume_btc, price_usd, transaction_volume_usd, utxo_count, total_btc, fees_usd, utxo_creation, daily_btc_mined, block_rewards_usd, cost_per_btc, asic_efficiency_jth, daily_electricity_kwh, daily_electricity_cost, electricity_cost_per_btc
  Período: 2015-01-01 00:00:00 a 2026-04-05 00:00:00


---

## 7. Documentação — Definição das Métricas e Limitações

### Definição Detalhada das Métricas

| Coluna | Métrica API | Unidade | Descrição |
|---|---|---|---|
| `transactions` | `n-transactions` | count/dia | Número total de transações confirmadas na blockchain Bitcoin por dia. Inclui todas as transações (pagamentos, consolidações, etc.) |
| `active_addresses` | `n-unique-addresses` | count/dia | Contagem de endereços únicos que apareceram como input ou output em transações confirmadas naquele dia |
| `hashrate` | `hash-rate` | TH/s | Taxa de hash estimada da rede Bitcoin. Calculada indiretamente a partir da dificuldade e do tempo entre blocos. Indica o poder computacional total dedicado à mineração |
| `difficulty` | `difficulty` | - | Parâmetro de dificuldade da rede. Ajustado automaticamente a cada ~2016 blocos (~2 semanas) para manter o tempo médio de bloco em ~10 minutos |
| `miners_revenue` | `miners-revenue` | USD/dia | Receita total dos mineradores = (recompensa de bloco × preço BTC) + taxas de transação em USD |
| `transaction_volume_btc` | `estimated-transaction-volume` | BTC/dia | Volume estimado de BTC transacionado on-chain. **Estimativa** que tenta excluir change outputs |

### Limitações dos Dados

1. **`transaction_volume_btc` é uma proxy (estimativa)**
   - O Bitcoin usa o modelo UTXO onde cada transação gera "troco" (change output). A Blockchain.com aplica heurísticas para estimar quanto do valor transferido é "real" vs. "troco"
   - Essas heurísticas não são perfeitas — o volume pode ser super ou subestimado
   - Transações CoinJoin, batching de exchanges e PayJoin distorcem a estimativa

2. **`active_addresses` ≠ usuários únicos**
   - Um único usuário pode controlar múltiplos endereços
   - Exchanges e serviços consolidam milhões de usuários em poucos endereços
   - Adoção de SegWit e endereços Taproot pode mudar padrões de contagem

3. **`hashrate` é estimado, não medido diretamente**
   - Calculado retrospectivamente a partir da dificuldade e tempo entre blocos
   - Sujeito a variância — blocos podem ser encontrados mais rápido/devagar que o esperado
   - Valores diários podem oscilar significativamente

4. **`difficulty` muda discretamente**
   - A dificuldade só muda a cada ~2016 blocos (~14 dias)
   - O forward fill é adequado pois o valor realmente permanece constante entre ajustes

5. **`miners_revenue` depende do preço BTC**
   - A receita em USD varia com o preço do Bitcoin, não apenas com atividade de mineração
   - Para análise de "atividade de mineração pura", é melhor usar receita em BTC

6. **Latência e disponibilidade da API**
   - A API é gratuita e sem autenticação — não há SLA de disponibilidade
   - Dados do dia mais recente podem estar incompletos
   - Rate limiting informal — pausas entre requests são recomendadas

7. **Dados antes de 2015**
   - Algumas métricas podem ter gaps ou menor granularidade nos primeiros anos do Bitcoin
   - `n-unique-addresses` pode ter contagem diferente em períodos muito antigos